In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
# Load dataset
train = datasets.MNIST(root="./data", train=True, download=True)
test = datasets.MNIST(root="./data", train=False, download=True)

In [3]:
# Calculate sample statistics (training set statistics)
standarized_train = train.data.float() / 255.0
mean_train = standarized_train.mean()
std_train = standarized_train.std()
print(f"Mean = {mean_train}\t\tStd Deviation = {std_train}")

Mean = 0.13066048920154572		Std Deviation = 0.30810782313346863


In [4]:
# Define transformations to be applied to the images
transformation = v2.Compose([
    # Convert to tensors since original MNIST images are in PIL (Pillow) format
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),

    # Normalize using sample statistics to prevent data leakage (training set statistics)
    # Normalize() takes an iterable, so the 0d tensor must be converted to 1d aka a list of 1 element
    v2.Normalize(mean=mean_train.view(1), std=std_train.view(1)),
])

In [5]:
# Apply transformations
train.transform = transformation
test.transform = transformation

In [6]:
# Create the loader
# batch_size: how many images to see at once
# shuffle: True ensures random order every epoch
train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False) # No need to shuffle test data

In [ ]:
# Mk1
class MNISTBaseline(nn.Module):
    def __init__(self):
        super(MNISTBaseline, self).__init__()
        # Define the layers
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # 16 channels * 14x14 image size after one poolings
        self.fc1 = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        # Define the flow (the logic)
        x = self.pool(F.relu(self.conv1(x)))

        x = x.view(-1, 16 * 14 * 14)  # Flattening the tensor
        x = self.fc1(x)
        return x

In [8]:
# # Mk2
# class MNISTBaseline(nn.Module):
#     def __init__(self):
#         super(MNISTBaseline, self).__init__()
#         # Define the layers
#         self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
#         self.pool = nn.MaxPool2d(2, 2)

#         # 32 channels * 7x7 image size after two poolings
#         self.fc1 = nn.Linear(32 * 7 * 7, 10)

#     def forward(self, x):
#         # Define the flow (the logic)
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))

#         x = x.view(-1, 32 * 7 * 7)  # Flattening the tensor
#         x = self.fc1(x)
#         return x

In [9]:
# # Mk3
# class MNISTBaseline(nn.Module):
#     def __init__(self):
#         super(MNISTBaseline, self).__init__()
#         # Layer 1: 28x28 -> 14x14
#         self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
#         # Layer 2: 14x14 -> 7x7
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
#         # Layer 3: 7x7 -> 3x3 (28 / 2 / 2 / 2 = 3.5, which rounds down to 3)
#         self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
#         self.pool = nn.MaxPool2d(2, 2)

#         # 64 channels * 3x3 final image size
#         self.fc1 = nn.Linear(64 * 3 * 3, 10)

#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x))) # Block 1
#         x = self.pool(F.relu(self.conv2(x))) # Block 2
#         x = self.pool(F.relu(self.conv3(x))) # Block 3

#         x = x.view(-1, 64 * 3 * 3)
#         x = self.fc1(x)
#         return x

In [10]:
# Define evaluation function
def evaluate(model, loader):
    model.eval() # Rule 1
    correct = 0
    total = 0
    
    with torch.no_grad(): # Rule 2
        for images, labels in loader:
            outputs = model(images)
            
            # The model outputs 10 raw numbers (logits). 
            # We take the index of the highest number as the prediction.
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f"Correct = {correct} / Total = {total}")
    print(f"Incorrect = {total - correct}")
    return accuracy, correct, total

In [11]:
import datetime
import torch


def log_model_auto(file_path, transform_obj, model, epochs, train_loss):
    """
    Automated logger that extracts details directly from PyTorch objects.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 1. Automatically extract transform details
    # This loops through your Compose list and grabs the string representation of each
    t_details = "\n".join([f"  - {str(t)}" for t in transform_obj.transforms])

    # 2. Extract Model details
    model_structure = str(model)
    total_params = sum(p.numel()
                       for p in model.parameters() if p.requires_grad)

    # 3. Evaluate the model
    test_acc, correct, total = evaluate(model, test_loader)

    # 4. Create the log entry
    log_entry = (
        f"\n{'='*60}\n"
        f"SESSION: {timestamp}\n"
        f"{'='*60}\n"
        f"[TRANSFORMS]\n{t_details}\n\n"
        f"[MODEL ARCHITECTURE]\n{model_structure}\n"
        f"Number of Epochs: {epochs}\n"
        f"Total Trainable Params: {total_params:,}\n\n"
        f"[RESULTS]\n"
        f"  Final Training Loss: {train_loss:.4f}\n"
        f"  Test Accuracy:       {test_acc:.2f}%\n"
        f"  Correct: {correct}\t\tIncorrect: {total - correct}\n"
        f"{'='*60}\n"
    )

    with open(file_path, "a") as f:
        f.write(log_entry)

In [12]:
# Initialize model, loss function and optimizer
model = MNISTBaseline()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
total_train_loss = 0.0
epochs = 1

model.train()       # Set model to training mode

# The Training Loop
for epoch in range(epochs):  # Run through the whole dataset 5 times
    epoch_loss = 0.0  # Reset for each epoch

    for images, labels in train_loader:
        # Clear the old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass & Optimize
        loss.backward()
        optimizer.step()

        # We add the loss of the current batch
        # .item() is CRITICAL to prevent memory leaks!
        epoch_loss += loss.item()

    # Calculate the average for this specific epoch
    avg_epoch_loss = epoch_loss / len(train_loader)
    # Update the global tracking variable for the logger
    total_train_loss = avg_epoch_loss

    print(f"Epoch {epoch+1} completed. Avg Loss: {avg_epoch_loss:.4f}")

# Log
log_model_auto(
    file_path="mnist_experiments.log",
    transform_obj=transformation,
    model=model,
    epochs=epochs,
    train_loss=total_train_loss
)

Epoch 1 completed. Avg Loss: 0.2036
Correct = 9716 / Total = 10000
Incorrect = 284


In [14]:
# Usage
test_acc, correct, total = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.2f}%")

Correct = 9716 / Total = 10000
Incorrect = 284
Test Accuracy: 97.16%


In [15]:
import matplotlib.pyplot as plt

def test_range(model, dataset, start_idx, end_idx):
    model.eval()
    results = []
    
    # We use torch.no_grad for efficiency reasons
    with torch.no_grad():
        for i in range(start_idx, end_idx):
            # Get the image and label from the dataset
            image, label = dataset[i]
            
            # Add a "batch" dimension because the model expects [1, 1, 28, 28]
            input_tensor = image.unsqueeze(0) # TODO
            
            # Get prediction
            output = model(input_tensor)
            _, predicted = torch.max(output, 1)
            
            results.append({
                'index': i,
                'actual': label,
                'predicted': predicted.item()
            })
            
    return results

# Example: Test the first 5 images of the test set
my_results = test_range(model, test, 0, 5)

for res in my_results:
    print(f"Image {res['index']} | Actual: {res['actual']} | Predicted: {res['predicted']}")

Image 0 | Actual: 7 | Predicted: 7
Image 1 | Actual: 2 | Predicted: 2
Image 2 | Actual: 1 | Predicted: 1
Image 3 | Actual: 0 | Predicted: 0
Image 4 | Actual: 4 | Predicted: 4


In [16]:
# Quick visualization snippet
def plot_test_results(model, dataset, num_images=5):
    model.eval()
    plt.figure(figsize=(10, 2))
    
    for i in range(num_images):
        img, label = dataset[i]
        output = model(img.unsqueeze(0))
        _, pred = torch.max(output, 1)
        
        plt.subplot(1, num_images, i+1)
        plt.imshow(img.squeeze(), cmap='gray')
        plt.title(f"P: {pred.item()} / A: {label}")
        plt.axis('off')
    plt.show()